<div style="background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 40px; border-radius: 12px; color: white; margin-bottom: 20px;">
  <h1 style="margin: 0; font-size: 2em;">🔍 Lab 04: Trace Your MAF Agent</h1>
  <p style="margin: 8px 0 0 0; font-size: 1.15em; opacity: 0.92;">Zava Bank Workshop — Hosted Agents with Microsoft Agent Framework</p>
</div>

## What We Learn

MAF agents get **auto-instrumented tracing** out of the box, and you can layer on custom spans for domain-specific context.

Tracing in the MAF path is especially valuable because tools execute **in-process**. That means the trace captures:

- Actual function execution time (not just a tool-call/response boundary)
- DataFrame operations inside the tool
- Any exceptions or edge cases within your Python code

In FSI production scenarios this granularity feeds performance optimization, compliance auditing, and incident investigation.

---
## 1 · Setup

In [ ]:
import os, json, pathlib
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

for key in ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_DEPLOYMENT"]:
    assert os.getenv(key), f"Missing env var: {key}"

# Data
DATA_DIR = pathlib.Path("../../data/zava-bank")
portfolios_df = pd.read_csv(DATA_DIR / "client_portfolios.csv")
risk_df       = pd.read_csv(DATA_DIR / "risk_metrics.csv")
market_df     = pd.read_csv(DATA_DIR / "market_data.csv")

print("✅ Environment and data loaded")

---
## 2 · Configure OpenTelemetry

We start with a **ConsoleSpanExporter** so you can see every span printed in the notebook. Later we'll switch to Azure Monitor.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor,
    ConsoleSpanExporter,
)

# Enable MAF internal tracing
os.environ["AZURE_AGENTS_TRACING_ENABLED"] = "true"
os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"

# Console exporter — spans print directly in the cell output
provider = TracerProvider()
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("zava-bank-maf-lab04")

print("✅ OpenTelemetry configured with ConsoleSpanExporter")

---
## 3 · Define Tools & Agent

Same tool functions from Lab 03, wired into the advisor agent.

In [ ]:
from typing import Annotated
from microsoft.agents.builder import BaseAgent, AgentContext


def search_portfolio(
    client_name: Annotated[str, "Client name to search for"] = None,
    ticker: Annotated[str, "Stock ticker symbol"] = None,
    sector: Annotated[str, "Sector to filter by"] = None,
) -> Annotated[str, "JSON array of matching portfolio positions"]:
    """Search Zava Bank client portfolio holdings."""
    results = portfolios_df.copy()
    if client_name:
        results = results[results["client_name"].str.lower().str.contains(client_name.lower())]
    if ticker:
        results = results[results["ticker"].str.upper() == ticker.upper()]
    if sector:
        results = results[results["sector"].str.lower().str.contains(sector.lower())]
    if results.empty:
        return json.dumps({"message": "No holdings found."})
    return results.to_json(orient="records")


def assess_risk(
    client_name: Annotated[str, "Client name"] = None,
    risk_tolerance: Annotated[str, "Filter by risk tolerance level"] = None,
) -> Annotated[str, "JSON array of risk metrics"]:
    """Get risk assessment for a client portfolio."""
    results = risk_df.copy()
    if client_name:
        results = results[results["client_name"].str.lower().str.contains(client_name.lower())]
    if risk_tolerance:
        results = results[results["risk_tolerance"].str.lower() == risk_tolerance.lower()]
    if results.empty:
        return json.dumps({"message": "No risk data found."})
    return results.to_json(orient="records")


def get_market_data(
    category: Annotated[str, "Category: Index, Sector Performance, or Economic Indicator"] = None,
    name: Annotated[str, "Indicator name to search for"] = None,
) -> Annotated[str, "JSON array of market data"]:
    """Get current market data, sector performance, and economic indicators."""
    results = market_df.copy()
    if category:
        results = results[results["category"].str.lower().str.contains(category.lower())]
    if name:
        results = results[results["name"].str.lower().str.contains(name.lower())]
    if results.empty:
        return json.dumps({"message": "No market data found."})
    return results.to_json(orient="records")


class ZavaBankAdvisorWithTools(BaseAgent):
    tools = [search_portfolio, assess_risk, get_market_data]

    SYSTEM_PROMPT = """You are the Zava Bank Client Advisor with access to real financial data.
Use your tools to answer questions:
- search_portfolio: Look up client holdings
- assess_risk: Get risk metrics
- get_market_data: Get market conditions
Always ground your responses in the data returned by your tools."""

    async def on_message(self, context: AgentContext, message: str) -> str:
        response = await context.invoke_model(
            system_prompt=self.SYSTEM_PROMPT,
            user_message=message,
            tools=self.tools,
        )
        return response


advisor = ZavaBankAdvisorWithTools()
print("✅ Agent created with", len(advisor.tools), "tools")

---
## 4 · Run a Traced Query (Console Output)

Watch the cell output below — you'll see spans for:
1. The top-level model invocation
2. Each tool call (with function name, arguments, duration)
3. The final response assembly

Because tools run in-process, the trace captures the **actual Python execution time** of each function, not just a serialized call/response pair.

In [ ]:
query = "What is Margaret Chen's portfolio composition and risk profile?"

response = await advisor.on_message(
    context=advisor.create_context(),
    message=query,
)

print("\n" + "=" * 60)
print(f"🗣️ Query: {query}\n")
print(f"🤖 Response:\n{response}")

---
## 5 · Switch to Azure Monitor Exporter

For production use we send traces to Application Insights. This requires the `APPLICATIONINSIGHTS_CONNECTION_STRING` env var.

In [ ]:
from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter

conn_str = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")
if not conn_str:
    print("⚠️  APPLICATIONINSIGHTS_CONNECTION_STRING not set — skipping Azure Monitor exporter.")
    print("   Set this in your .env to send traces to Application Insights.")
else:
    azure_exporter = AzureMonitorTraceExporter(connection_string=conn_str)
    provider.add_span_processor(SimpleSpanProcessor(azure_exporter))
    print("✅ Azure Monitor exporter added — traces will appear in Application Insights")

---
## 6 · Custom FSI Spans

Layer domain-specific attributes onto your traces. In FSI these are useful for:
- **Compliance** — tag which client segment was queried
- **Performance** — identify slow tool calls by query type
- **Audit** — record which tools were expected vs. actually invoked

In [ ]:
query_fsi = "Review Apex Capital's portfolio and compare their risk exposure to current market conditions."

with tracer.start_as_current_span("zava-maf-portfolio-review") as span:
    span.set_attribute("finance.client_segment", "institutional")
    span.set_attribute("finance.query_type", "portfolio_review")
    span.set_attribute("finance.tool_calls_expected", "search_portfolio,assess_risk,get_market_data")
    span.set_attribute("zava.agent_type", "maf_hosted")
    span.set_attribute("zava.lab", "04-tracing")

    response_fsi = await advisor.on_message(
        context=advisor.create_context(),
        message=query_fsi,
    )

    span.set_attribute("finance.response_length", len(response_fsi))

print(f"🗣️ Query: {query_fsi}\n")
print(f"🤖 Response:\n{response_fsi}")

---
## MAF Tracing vs Prompt Agent Tracing

| Aspect | Prompt Agent (Path A) | MAF Agent (Path B) |
|---|---|---|
| **Tool visibility** | Tool call + tool response boundaries | Full in-process function execution span |
| **Timing accuracy** | Includes network overhead to/from tool endpoint | Actual Python function wall time |
| **Custom attributes** | Same — you can add domain spans in both paths | Same — you can add domain spans in both paths |
| **Exception capture** | Platform-level error | Stack trace from your function |
| **FSI value** | Good — see what the model asked for | Better — see exactly what data was accessed and how long each lookup took |

In production FSI deployments this granularity matters. When a compliance officer asks "which client records were accessed during this session?", the MAF trace can answer that from span attributes alone.

---
## 7 · Cleanup

In [ ]:
# Flush pending spans
provider.force_flush()

del advisor
print("🧹 Cleanup complete — spans flushed")

---
<div style="background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 20px 30px; border-radius: 10px; color: white;">
  <h2 style="margin: 0 0 8px 0;">✅ Lab 04 Summary</h2>
  <ul style="margin: 0; padding-left: 20px; line-height: 1.8;">
    <li>MAF auto-instruments agent interactions — enable with one env var.</li>
    <li>In-process tool execution produces <strong>deeper trace spans</strong> than prompt agents.</li>
    <li>Custom FSI attributes (<code>finance.*</code>, <code>zava.*</code>) make traces queryable for compliance and audit.</li>
    <li>Console exporter is great for development; Azure Monitor exporter for production.</li>
  </ul>
  <p style="margin: 12px 0 0 0; opacity: 0.85;">In production FSI deployments, this tracing granularity feeds performance optimization, compliance auditing, and incident investigation.</p>
</div>